# Введення в PyTorch і персептрон


Персептрон є найпростішою формою нейронної мережі.

Нейронна мережа - утворення, що складється з вхідного шару (чутливі елементи, сенсори, S-елементи), вихідного шару (виконавчі, R-елементи) та певної кількості проміжних шарів (асоціативних, прихованих, A-елементів).

Кожен з елементів може знаходитись в одному з станів: активному чи неактивному (1 або 0). Активні елементи є джерелами сигналів для наступного шару, причому по різних зв'язках сигнал передається з різним коефіцієнтом (вагою зв'язку) - довільним дійсним числом. Навпаки, сигнали, що надходять до одного елементу за різними зв'язками, додаються (множені на коефіцієнти) та порівнюються з певним пороговим значенням, яке активує або деактивує відповідний елемент. Сукупність коефіцієнтів всієї мережі - визначає її роботу і є "знаннями" мережі.

Значення коефіцієнтів можуть бути кореговані в результаті навчання перцептрону:
- на вхід подається тестові сигнали різних типів
- елементам, які активуються при цьому, змінюють вагу у відповідності до типу сигналу:
    - для сигналів одного типу -- збільшують на 1
    - для сигналів іншого типу -- зменшують на 1
Як результат, будуть залишатись великі (за модулем) ваги лише для "різницевих" елементів, які активуються лише одним типом сигналів, а для тих, що активуються обома типами, сумарний результат буде близькій до нуля.

Інший тип навчання - зворотнє поширення помилки. Якщо мережа дає неправильний результат, то різниця правильного та неправильного сигналу подається зворотнім шляхом і кожен коефіцієнт виправляється "градієнтом". Для цього елементи повинні запам'ятовувати прийняті рішення, щоб мати можливість відкоригувати їх при зворотньому ході.

## Бізнес-контекст
Компанія HealthRisk Analytics будує прототип моделі, яка за простими показниками (наприклад, вік та індекс маси тіла) прогнозує рівень страхового ризику клієнта.
Ваше завдання — змоделювати основу цієї системи, використовуючи PyTorch: спочатку — один лінійний нейрон, потім — малу нейромережу.
## Завдання 1
Створіть тензори та виконайте базові операції.
- Уявіть, що кожен тензор — це стовпець даних про клієнтів (вік, вага, зріст).
- Створіть 1D, 2D і 3D тензори(torch.manual_seed(42)).
- Виконайте операції додавання і поелементного множення, щоб обчислити умовний показник ризику.
- Увімкніть requires_grad=True для одного тензора, перевірте наявність grad_fn.

In [1]:
import torch

In [3]:
torch.manual_seed(42)  # сідування для відтворюваності даних
age = torch.randint(18, 70, (5,), dtype=torch.float32)
# даний генератор дозволяє встановити межі, проте, єдині для всіх
age

tensor([36., 65., 50., 48., 48.])

In [8]:
# у випадках коли потрібні різні діапазони (мінімальні та максимальні значення)
# використовується нормований генератор з наступним обробленням даних 
age = torch.randn(5)
print("Початкові величини (нормальний розподіл):", age)
age = 44.0 + 26.0 * age
print("Після оброблення:", age)

Початкові величини (нормальний розподіл): tensor([-0.0131,  1.5860, -1.0186, -0.4180, -0.2935])
Після оброблення: tensor([43.6606, 85.2353, 17.5161, 33.1312, 36.3690])


In [20]:
# трьохпараметричний тензор 
age_weight_height = torch.randn(5,3, 
                requires_grad=True)  # requires_grad - включення "пам'яті" змін
age_weight_height

tensor([[-0.1955, -0.6541,  0.3254],
        [ 1.1969,  0.5302,  0.6729],
        [ 0.5023, -1.6336,  0.7949],
        [-0.4340,  0.1795,  0.0356],
        [-0.5065, -1.2639, -0.1908]], requires_grad=True)

In [21]:
with torch.no_grad():  # операції самим з собою вимагають відключення пам'яті
    age_weight_height[:,0] = 44.0  + 16.0 * age_weight_height[:,0]
    age_weight_height[:,1] = 95.0  + 25.0 * age_weight_height[:,1]   # 45..145:  95 +- 50
    age_weight_height[:,2] = 180.0 + 15.0 * age_weight_height[:,2]   # 150..210: 180 +- 30
age_weight_height

tensor([[ 40.8712,  78.6469, 184.8811],
        [ 63.1505, 108.2551, 190.0938],
        [ 52.0367,  54.1603, 191.9235],
        [ 37.0561,  99.4863, 180.5339],
        [ 35.8963,  63.4022, 177.1376]], requires_grad=True)

In [22]:
# Виконайте операції додавання і поелементного множення, щоб обчислити умовний показник ризику.
risk = age_weight_height[:,0] * age_weight_height[:,1] + 0.5 * age_weight_height[:,2]
risk = (risk - risk.min()) / (risk.max() - risk.min())
risk

tensor([0.2063, 1.0000, 0.1204, 0.3093, 0.0000], grad_fn=<DivBackward0>)

## Завдання 2
Реалізуйте персептрон для передбачення ризику клієнта.
- Згенеруйте дані (torch.randn, 100 зразків, 1 ознака — індекс маси тіла).
- Цільова змінна — умовний "ризик" (лінійна комбінація + шум).
- Побудуйте модель nn.Linear(1, 1) і функцію втрат MSE.
- Виконайте 20 епох навчання з ручним оновленням параметрів.
- Виведіть початкові та фінальні значення ваг і loss.

In [27]:
# Згенеруйте дані (torch.randn, 100 зразків, 1 ознака — індекс маси тіла).
bmi = torch.randn(100,1)
print(bmi[:5])
# Цільова змінна — умовний "ризик" (лінійна комбінація + шум).
noise = 0.1 * torch.randn(100,1)
risk = 0.1 * bmi + noise
print(risk[:5])

tensor([[-1.3095],
        [-0.4104],
        [ 0.4681],
        [-0.2346],
        [ 0.2782]])
tensor([[-0.1545],
        [-0.2792],
        [ 0.1201],
        [-0.1348],
        [-0.0156]])


In [28]:
# Побудуйте модель nn.Linear(1, 1) і функцію втрат MSE.
import torch.nn as nn
import torch.optim as optim

class Perceptron(nn.Module) :
    def __init__(self) :
        super(Perceptron, self).__init__()
        self.layer1 = nn.Linear(1,1)
        self.activator = nn.Sigmoid()

    def forward(self, x) :
        x = self.layer1(x)
        x = self.activator(x)
        return x

loss_f = nn.functional.mse_loss  # функція втрат MSE.

In [33]:
# Виконайте 20 епох навчання з ручним оновленням параметрів.
model = Perceptron()
learning_rate = 0.1  # Ступінь коригування параметрів на помилку
print(*model.parameters())
# Цикл навчання
for epoch in range(20) :  # Цикли навчання називають епохами
    output = model(bmi)   # Результат проходження "сигналу" мережею (перцептроном)
    loss = loss_f(output, risk)  # Визначаємо відхилення одержаного результату від бажаного
    loss.backward()       # Зворотнє поширення помилки - визначення градієнтів відхилення
    # ручне оновлення параметрів
    with torch.no_grad() :
        for param in model.parameters() :         # param - ваговий коефіцієнт (вага зв'язку)
            param -= learning_rate * param.grad   # -= -- повертаємо параметр з урахуванням навчального рівня
            param.grad.zero_()                    # скидаємо врахований градієнт
    print(f"Epoch: {epoch}, loss: {loss.item():.4f}")        
    

Parameter containing:
tensor([[0.6424]], requires_grad=True) Parameter containing:
tensor([0.2477], requires_grad=True)
Epoch: 0, loss: 0.3234
Epoch: 1, loss: 0.3172
Epoch: 2, loss: 0.3112
Epoch: 3, loss: 0.3052
Epoch: 4, loss: 0.2993
Epoch: 5, loss: 0.2935
Epoch: 6, loss: 0.2878
Epoch: 7, loss: 0.2822
Epoch: 8, loss: 0.2767
Epoch: 9, loss: 0.2713
Epoch: 10, loss: 0.2660
Epoch: 11, loss: 0.2608
Epoch: 12, loss: 0.2557
Epoch: 13, loss: 0.2507
Epoch: 14, loss: 0.2457
Epoch: 15, loss: 0.2409
Epoch: 16, loss: 0.2362
Epoch: 17, loss: 0.2316
Epoch: 18, loss: 0.2270
Epoch: 19, loss: 0.2226
